# Musanga V4 — NB02: paired RGB experiment (circles only versus circles + plots), v4_003

**Stage 1: CPU-only supervised-reference and full-chip leakage audit.** First make a *single* authoritative split and training sample manifest for both RGB experiments, inspect coverage/contradictions, and then authorize GPU training in a subsequent NB02 revision. This notebook deliberately contains **no fitting**: it stops before GPU costs, following the earlier V4 debugging experience.

**Primary comparison:** Model A = original KVO-final exhaustive circles + site-specific partial tree annotations; Model B = *identical* inputs plus 78 Nadine exhaustive plot units (respecting their 50-m spatial grouping). All 30 retained circles and 78 plots are candidate outer-fold evaluation units; both models use the exact same held-out fold and assessment support. The historical V3.1 30-circle folds remain separately available as a retrospective benchmark, not interchangeable with this redesigned fold structure.

**Sources:** V3.1 audited reference and original ResNet18 U-Net recipe; V4 NB00 78 annotated plots and 3,324 crowns; V4 NB01 v4_002 frozen 50-m grouped / cover-balanced folds. Strictly distinguish mapped crown cover from field basal area.

**Policies:** Do not treat unknown pixels as negative. Within a circle, use KVO-final exhaustive Musanga polygons; within a non-circle plot, use Nadine exhaustive Musanga polygons. Negative non-Musanga tree crowns can also inform partial supervision. When the two reference systems disagree in overlapping areas, mask conflict pixels and report their geometry. Do not count intersecting references as independent validation replicates. For every fold, exclude *full training windows* touching held-out polygon + buffer, across all mosaics and all annotation sources. RGB training must not require DSM/MS coverage.


## 00 — Inputs, frozen split contract and reference provenance

Reads only metadata and vector files. Requires NB00 v4_001 Section 09 and NB01 v4_002 Sections 00–03 executed successfully. Input files and previous outputs are read-only.


In [1]:
%pip -q install geopandas pyogrio shapely pandas numpy matplotlib rasterio
from pathlib import Path
import json,math,hashlib
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')] if p.exists()),None)
assert ROOT is not None
DATA=ROOT/'Data'
OLD=ROOT/'Outputs/V3_1'
AUDIT=ROOT/'Outputs/v4/00_input_audit'
SPLIT_ROOT=ROOT/'Outputs/v4/01_spatial_cover_folds_v2'
OUT=ROOT/'Outputs/v4/02_rgb_circles_plus_plots_v1'
OUT.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
SPLIT_CONTRACT=json.loads((SPLIT_ROOT/'01_reference_fold_contract_v2.json').read_text())
FOLD_CSV=Path(SPLIT_CONTRACT['output_csv'])
assert FOLD_CSV.exists(),FOLD_CSV
assert hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()==SPLIT_CONTRACT['csv_sha256'], 'NB01 fold file changed'
SPLITS=pd.read_csv(FOLD_CSV)
assert len(SPLITS)==108 and set(SPLITS.unit_type)=={'circle','plot'}
assert (SPLITS.unit_type=='circle').sum()==30 and (SPLITS.unit_type=='plot').sum()==78
assert SPLITS.groupby('spatial_group').fold.nunique().eq(1).all()
assert set(SPLITS.fold)=={0,1,2}
GEOMS=gpd.read_file(SPLIT_ROOT/'01_reference_groups_and_folds_v2.gpkg').to_crs('EPSG:32635')
assert len(GEOMS)==108
assert not GEOMS.duplicated(['unit_type','unit_id']).any()
SITES=['Baego','MOS','Yangambi','Yoko']
assert set(GEOMS.site)==set(SITES)
PLOT_PATH=Path(F78['plot_file'])
CROWN_PATH=Path(F78['exhaustive_plot_crowns'])
EXH_PATH=Path(F78['circle_positive_crowns'])
CIRCLE_PATH=Path(F78['circle_complete_roi'])
ROI_PATH=Path(V31['roi'])
for p in [PLOT_PATH,CROWN_PATH,EXH_PATH,CIRCLE_PATH,ROI_PATH]:assert p.exists(),p
print('REFERENCE:',len(GEOMS),'units;',GEOMS.spatial_group.nunique(),'spatial groups;',
      GEOMS.groupby('fold').size().to_dict(),'units by outer fold')
print('SPLIT FINGERPRINT:',SPLIT_CONTRACT['csv_sha256'][:20])
print('V3.1 KVO polygons:',EXH_PATH.name,'| V4 Nadine crowns:',CROWN_PATH.name)
print('OUT (new only):',OUT)


Mounted at /content/drive
REFERENCE: 108 units; 55 spatial groups; {0: 33, 1: 38, 2: 37} units by outer fold
SPLIT FINGERPRINT: 8c24868ff48350e389a3
V3.1 KVO polygons: Circles_Annotations_kvo_final.shp | V4 Nadine crowns: Tree_Annotation_georef_nadine_new 1.gpkg
OUT (new only): /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/v4/02_rgb_circles_plus_plots_v1


## 01 — Source annotation geometry, reference priority, and overlap conflict audit

**Source semantics:** V3.1 final exhaustive circle polygons are Musanga positives; inside an exhaustive circle the remaining area is certified non-Musanga. The V4 crown file has `Class_ID == 1` as Musanga; other classes are explicit negative crowns; the audited plot field has `annotated = True`. The latter does **not** by itself prove all unannotated background is biologically negative, so the assumption is recorded prominently for verification.

Compute overlap disagreements (positive in circle versus negative in plot, and vice versa) *by actual polygon area*, without reading large RGB mosaics. **Conflict pixels must be masked for Model B**, not silently relabelled. The circle reference remains the authoritative label for overlapping reference areas only where there is no contradiction.


In [2]:
# NB02.01 — Topology-safe vector audit; fail closed on material geometry changes.
from shapely.validation import explain_validity
from shapely import make_valid,union_all,set_precision
from shapely.geometry import GeometryCollection
from shapely.ops import unary_union
REF_CRS='EPSG:32635'
circles_full=gpd.read_file(CIRCLE_PATH).to_crs(REF_CRS)
circles_full['unit_id']=circles_full.index.astype(str)
circles=circles_full[circles_full.unit_id.isin(
    GEOMS.loc[GEOMS.unit_type.eq('circle'),'unit_id'])].copy()
plots=gpd.read_file(PLOT_PATH).to_crs(REF_CRS)
plots['unit_id']=plots.Plot_name.astype(str)
assert len(circles)==30 and len(plots)==78
assert plots.unit_id.is_unique and circles.unit_id.is_unique

def polygon_parts(geom):
    if geom is None or geom.is_empty:return []
    if geom.geom_type in ('Polygon','MultiPolygon'):return [geom]
    return [part for child in getattr(geom,'geoms',[]) for part in polygon_parts(child)]

QA02=[]
def repair_polygon_geoms(geoms,source,maximum_relative_change=.05):
    fixed=[]
    for ix,geom in enumerate(geoms):
        if geom is None or geom.is_empty:
            QA02.append(dict(source=source,row=ix,status='EMPTY_STOP',relative_area_change=np.nan,
                             original_valid=False,reason='empty'))
            raise ValueError(f'{source} polygon {ix} is empty; cannot silently drop reference')
        original_valid=bool(geom.is_valid)
        reason='valid' if original_valid else explain_validity(geom)
        candidate=geom if original_valid else make_valid(geom)
        parts=polygon_parts(candidate)
        if not parts:
            QA02.append(dict(source=source,row=ix,status='NO_POLYGON_STOP',relative_area_change=np.nan,
                             original_valid=original_valid,reason=reason))
            raise ValueError(f'{source} {ix} cannot be repaired into polygons: {reason}')
        try:
            candidate=unary_union(parts) if len(parts)>1 else parts[0]
        except Exception as exc:
            raise ValueError(f'Failed polygon repair for {source}, row {ix}: {exc}') from exc
        if not candidate.is_valid or candidate.is_empty:
            raise ValueError(f'Invalid repaired geometry: {source} row {ix}')
        # Invalid input polygon area is GEOS-defined but may be ambiguous; audit both.
        original_area=float(geom.area)
        new_area=float(candidate.area)
        change=abs(new_area-original_area)/max(abs(new_area),abs(original_area),1.e-9)
        status='unchanged' if original_valid else 'repaired'
        QA02.append(dict(source=source,row=ix,status=status,
            relative_area_change=change,original_valid=original_valid,reason=reason))
        if change>maximum_relative_change:
            raise ValueError(
                f'{source} row {ix}: repair changes polygon area by {change:.2%} '
                f'(limit {maximum_relative_change:.0%}). Review original shape manually.')
        fixed.append(candidate)
    return fixed

def robust_union02(geoms,source):
    geoms=[q for q in geoms if q is not None and not q.is_empty]
    if not geoms:return GeometryCollection()
    try:
        out=unary_union(geoms)
    except Exception as exc:
        print(f'WARNING {source}: exact union failed ({type(exc).__name__}); '
              'retrying 1-cm precision grid after individual geometry validation.')
        try:
            out=union_all(geoms,grid_size=.01)
        except Exception as exc2:
            raise RuntimeError(f'{source}: union failed at exact and 1-cm precision. '
                               'Review topological conflicts before training.') from exc2
    if not out.is_valid:
        raise ValueError(f'{source}: union is invalid; do not use for labels')
    return out

# Check source polygons BEFORE attempting any union. This is the fix for the
# TopologyException in the raw KVO exhaustive-positive shapefile.
exh=gpd.read_file(EXH_PATH).to_crs(REF_CRS)
exh.geometry=repair_polygon_geoms(exh.geometry,'V3_1_exhaustive_Musanga')
crowns=gpd.read_file(CROWN_PATH).to_crs(REF_CRS)
crowns['is_musanga']=pd.to_numeric(crowns.Class_ID,errors='raise').eq(1)
crowns.geometry=repair_polygon_geoms(crowns.geometry,'V4_Nadine_plot_crowns')
circles.geometry=repair_polygon_geoms(circles.geometry,'V3_1_circle_boundaries')
plots.geometry=repair_polygon_geoms(plots.geometry,'V4_plot_boundaries')
print('EXHAUSTIVE REFERENCE REPAIR SUMMARY (no rows dropped):')
display(pd.DataFrame(QA02).groupby(['source','status']).agg(
    n=('row','size'),max_rel_area_change=('relative_area_change','max')).round(5))
pd.DataFrame(QA02).to_csv(OUT/'02_source_geometry_repairs.csv',index=False)
print('Saved per-polygon repair audit:',OUT/'02_source_geometry_repairs.csv')

# Partial crown data can be quarantined under a separate, documented policy:
# a partial annotation is OPTIONAL supplementary supervision; a retained exhaustive
# circle or plot is NOT optional and must continue to fail closed on material repairs.
# Keep source-row indices intact so exclusions are traceable.
partial={}
partial_quarantine=[]
for site,path in V31['shared_annotations'].items():
    v=gpd.read_file(path).to_crs(REF_CRS)
    assert 'Class_ID' in v.columns
    v['musanga']=pd.to_numeric(v.Class_ID,errors='coerce').eq(1)
    retained=[]
    for source_row,geom in zip(v.index.tolist(),v.geometry.tolist()):
        label=f'V3_1_partial_{site}'
        original_valid=(geom is not None and not geom.is_empty and bool(geom.is_valid))
        reason=('valid' if original_valid else
                'empty' if geom is None or geom.is_empty else explain_validity(geom))
        change=np.nan
        try:
            if geom is None or geom.is_empty:
                raise ValueError('empty crown')
            candidate=geom if original_valid else make_valid(geom)
            parts=polygon_parts(candidate)
            if not parts:
                raise ValueError('no polygonal component after make_valid')
            candidate=unary_union(parts) if len(parts)>1 else parts[0]
            if candidate.is_empty or not candidate.is_valid:
                raise ValueError('invalid candidate after repair')
            change=abs(float(candidate.area)-float(geom.area))/max(
                float(candidate.area),float(geom.area),1e-9)
            if change>.05:
                raise ValueError(f'area change {change:.2%} exceeds 5%')
            retained.append((int(source_row),candidate))
            QA02.append(dict(source=label,row=int(source_row),
                status='unchanged' if original_valid else 'repaired',
                relative_area_change=change,original_valid=original_valid,reason=reason))
        except Exception as exc:
            QA02.append(dict(source=label,row=int(source_row),
                status='quarantined_partial',relative_area_change=change,
                original_valid=original_valid,reason=f'{reason}; {exc}'))
            partial_quarantine.append(dict(site=site,source_row=int(source_row),
                relative_area_change=change,original_valid=original_valid,
                reason=str(exc),source_path=str(path)))
    keep_ids=[idx for idx,_ in retained]
    v=v.loc[keep_ids].copy()
    v.geometry=[candidate for _,candidate in retained]
    assert v.geometry.is_valid.all()
    partial[site]=v
    print(f'PARTIAL {site}: retained {len(v)} / '
          f'{len(v)+sum(x["site"]==site for x in partial_quarantine)}; '
          f'quarantined {sum(x["site"]==site for x in partial_quarantine)}')
assert set(partial)==set(SITES)
assert all(len(v)>0 for v in partial.values()),'Missing usable partial crowns'
# Guard against a widespread underlying geometry/reference issue.
total_partial=sum(len(v) for v in partial.values())+len(partial_quarantine)
quarantine_rate=len(partial_quarantine)/total_partial
pd.DataFrame(QA02).to_csv(OUT/'02_source_geometry_repairs.csv',index=False)
pd.DataFrame(partial_quarantine,columns=[
    'site','source_row','relative_area_change','original_valid','reason','source_path'
]).to_csv(OUT/'02_partial_quarantine.csv',index=False)
print('PARTIAL QUARANTINE:',len(partial_quarantine),'/',total_partial,
      f'({quarantine_rate:.3%})')
display(pd.DataFrame(QA02).groupby(['source','status']).agg(
    n=('row','size'),max_rel_area_change=('relative_area_change','max')).round(5))
if quarantine_rate>.01:
    raise RuntimeError('More than 1% of partial crown polygons required quarantine. '
                       'Inspect 02_partial_quarantine.csv before proceeding.')

circle_union=robust_union02(list(circles.geometry),'circle footprints')
plot_union=robust_union02(list(plots.geometry),'plot footprints')
exh_union=robust_union02(list(exh.geometry),'KVO exhaustive-positive polygons')
circ_positive=exh_union.intersection(circle_union)
plot_pos=robust_union02(list(crowns.loc[crowns.is_musanga].geometry),
                        'Nadine positive crowns').intersection(plot_union)
plot_nonpos=robust_union02(list(crowns.loc[~crowns.is_musanga].geometry),
                           'Nadine negative crowns').intersection(plot_union)
shared=circle_union.intersection(plot_union)
# Compare definitions ONLY over their shared complete-annotation footprints.
circle_pos_shared=circ_positive.intersection(shared)
plot_pos_shared=plot_pos.intersection(shared)
diff_area=float(circle_pos_shared.symmetric_difference(plot_pos_shared).area)
shared_area=float(shared.area)
plot_internal_overlap=float(plot_pos.intersection(plot_nonpos).area)
print('Circle/plot complete-annotation overlap (m²):',round(shared_area,1))
print('Musanga label disagreement on overlap (m²):',round(diff_area,1))
print('Musanga disagreement % of shared area:',
      round(100*diff_area/shared_area,3) if shared_area else 0.)
print('Plot positive vs explicit negative crown overlap (m²):',
      round(plot_internal_overlap,2))
conflicts=circle_pos_shared.symmetric_difference(plot_pos_shared)
conflicts=conflicts.union(plot_pos.intersection(plot_nonpos))
assert conflicts.is_valid
reference_summary=dict(
  schema='musanga_v4_nb02_annotation_overlap_audit_v2',
  n_circles=30,n_plots=78,n_nadine_crowns=len(crowns),
  n_exhaustive_KVO_polygons=len(exh),
  source_geometry_repair_audit=str(OUT/'02_source_geometry_repairs.csv'),
  partial_quarantine_audit=str(OUT/'02_partial_quarantine.csv'),
  n_partial_quarantined=len(partial_quarantine),
  partial_quarantine_fraction=quarantine_rate,
  circle_plot_shared_area_m2=shared_area,
  exhaustive_positive_conflict_area_m2=diff_area,
  within_plot_positive_negative_overlap_m2=plot_internal_overlap,
  circle_authoritative='V3_1_KVO_final',
  plot_authoritative='V4_Nadine_exhaustive',
  conflict_policy='suppress contradicted supervised pixels (NOT relabel)',
  plot_background_negative_assumption='PENDING MANUAL SIGNOFF: annotated=True implies exhaustive canopy Musanga annotation')
(OUT/'02_reference_annotation_overlap_audit_v2.json').write_text(json.dumps(reference_summary,indent=2))
display(pd.DataFrame([reference_summary]).T.rename(columns={0:'value'}))
print('Review geometry repairs and overlap metrics before GPU training.')


EXHAUSTIVE REFERENCE REPAIR SUMMARY (no rows dropped):


n  max_rel_area_change
source                  status                              
V3_1_circle_boundaries  unchanged    30              0.00000
V3_1_exhaustive_Musanga repaired    254              0.04465
                        unchanged   381              0.00000
V4_Nadine_plot_crowns   repaired    523              0.02690
                        unchanged  2801              0.00000
V4_plot_boundaries      unchanged    78              0.00000

Saved per-polygon repair audit: /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/v4/02_rgb_circles_plus_plots_v1/02_source_geometry_repairs.csv
PARTIAL Baego: retained 890 / 890; quarantined 0
PARTIAL Yangambi: retained 969 / 969; quarantined 0
PARTIAL Yoko: retained 1910 / 1910; quarantined 0
PARTIAL MOS: retained 4990 / 4991; quarantined 1
PARTIAL QUARANTINE: 1 / 8760 (0.011%)


n  max_rel_area_change
source                  status                                        
V3_1_circle_boundaries  unchanged              30              0.00000
V3_1_exhaustive_Musanga repaired              254              0.04465
                        unchanged             381              0.00000
V3_1_partial_Baego      repaired               24              0.00669
                        unchanged             866              0.00000
V3_1_partial_MOS        quarantined_partial     1              0.05875
                        repaired               40              0.04821
                        unchanged            4950              0.00000
V3_1_partial_Yangambi   repaired                2              0.01075
                        unchanged             967              0.00000
V3_1_partial_Yoko       repaired                4              0.00000
                        unchanged            1906              0.00000
V4_Nadine_plot_crowns   repaired              523              0.02690
                        unchanged            2801              0.00000
V4_plot_boundaries      unchanged              78              0.00000

Circle/plot complete-annotation overlap (m²): 92301.5
Musanga label disagreement on overlap (m²): 2971.6
Musanga disagreement % of shared area: 3.219
Plot positive vs explicit negative crown overlap (m²): 18.36


,value
schema,musanga_v4_nb02_annotation_overlap_audit_v2
n_circles,30
n_plots,78
n_nadine_crowns,3324
n_exhaustive_KVO_polygons,635
source_geometry_repair_audit,/content/drive/MyDrive/Colab Notebooks/Musanga...
partial_quarantine_audit,/content/drive/MyDrive/Colab Notebooks/Musanga...
n_partial_quarantined,1
partial_quarantine_fraction,0.000114
circle_plot_shared_area_m2,92301.484595


Review geometry repairs and overlap metrics before GPU training.


## 02 — RGB mosaic ownership and full-chip spatial exclusion

Only RGB orthomosaics are used. Compare **all six** V3.1 inventory mosaics, including MOS 3–4 because some of the new plots may occur there. Determine each reference unit's valid *geometric* RGB mosaic coverage using its site and intersection area; do not silently drop or double-count references where mosaics overlap. The reference owner's mosaic is a deterministic maximum-footprint intersection, and coverage in ROI is reported.

Create a reproducible, bounded **256×256** candidate-chip grid for training units in each fold, then reject any candidate whose **entire image-chip footprint intersects a held-out plot/circle buffered by 50 m**, even across adjacent mosaics. This is a conservative test. Held-out units are never used as training labels, and partial crowns are also subject to this full-chip filter.

This stage is **vector/metadata only**, and outputs a sampling manifest; actual RGB reading, pixel supervision assembly, and train/validation chips come in the next step.


In [3]:
RGB_DIR=DATA/'Ortho_RGB'
mosaics={}
for d in V31['mosaics']:
    name=d['file']
    if name.endswith('.tif'):
        p=Path(d['path'])
        if not p.exists():p=RGB_DIR/name
        if not p.exists():continue
        with rasterio.open(p) as ds:
            assert ds.crs is not None and ds.count>=3
            bounds=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(REF_CRS).iloc[0]
            mosaics[name]=dict(path=p,site=str(d['site']),bounds=bounds,
                               pixel_size_m=max(abs(ds.transform.a),abs(ds.transform.e)))
assert len(mosaics)==6,[(k,v['site']) for k,v in mosaics.items()]
assert set(v['site'] for v in mosaics.values())==set(SITES)
roi=gpd.read_file(ROI_PATH).to_crs(REF_CRS)
roi_union=unary_union(list(roi.geometry))
units=GEOMS.copy()
owners=[]
for u in units.itertuples(index=False):
    candidates=[(name,float(u.geometry.intersection(d['bounds']).intersection(roi_union).area))
         for name,d in mosaics.items() if d['site']==u.site]
    name,overlap=max(candidates,key=lambda v:(v[1],v[0]))
    frac=overlap/max(u.geometry.area,1e-9)
    owners.append(dict(unit_type=str(u.unit_type),unit_id=str(u.unit_id),
       site=str(u.site),fold=int(u.fold),spatial_group=str(u.spatial_group),
       mosaic=name,roi_mosaic_coverage_fraction=frac))
OWNERS=pd.DataFrame(owners)
assert OWNERS[['unit_type','unit_id']].duplicated().sum()==0
display(OWNERS.groupby(['site','unit_type','mosaic']).agg(n=('unit_id','size'),
     min_roi_coverage=('roi_mosaic_coverage_fraction','min')).round(3))
print('Units with <95% geometric RGB/ROI coverage:',
      int(OWNERS.roi_mosaic_coverage_fraction.lt(.95).sum()))
display(OWNERS.loc[OWNERS.roi_mosaic_coverage_fraction.lt(.95)])
OWNERS.to_csv(OUT/'02_reference_mosaic_owners.csv',index=False)
# Exclusion geometry for *each fold*, irrespective of owner mosaic.
HELDOUT={}
for f in range(3):
    q=units.loc[units.fold.eq(f)]
    HELDOUT[f]={site:unary_union([geom.buffer(50.) for geom in q.loc[q.site.eq(site),'geometry']])
                for site in SITES}
assert all(len(units.loc[units.fold.eq(f)])>0 for f in range(3))
print('Full-chip exclusion geometry prepared; original NB01 spatial groups untouched.')


n  \
site     unit_type mosaic                                             
Baego    circle    Baego.tif                                      7   
         plot      Baego.tif                                      7   
MOS      circle    Mos_1_2_transparent_mosaic_group1_georef.tif   6   
                   Mos_5_6_transparent_mosaic_group1_georef.tif   5   
         plot      Mos_1_2_transparent_mosaic_group1_georef.tif   5   
                   Mos_3_4_transparent_mosaic_group1_georef.tif   5   
                   Mos_5_6_transparent_mosaic_group1_georef.tif  10   
Yangambi circle    Yangambi_georef.tif                            7   
         plot      Yangambi_georef.tif                           41   
Yoko     circle    Yoko_kvo_georef.tif                            5   
         plot      Yoko_kvo_georef.tif                           10   

                                                                 min_roi_coverage  
site     unit_type mosaic                                                          
Baego    circle    Baego.tif                                                1.000  
         plot      Baego.tif                                                1.000  
MOS      circle    Mos_1_2_transparent_mosaic_group1_georef.tif             1.000  
                   Mos_5_6_transparent_mosaic_group1_georef.tif             1.000  
         plot      Mos_1_2_transparent_mosaic_group1_georef.tif             1.000  
                   Mos_3_4_transparent_mosaic_group1_georef.tif             1.000  
                   Mos_5_6_transparent_mosaic_group1_georef.tif             0.000  
Yangambi circle    Yangambi_georef.tif                                      0.984  
         plot      Yangambi_georef.tif                                      1.000  
Yoko     circle    Yoko_kvo_georef.tif                                      1.000  
         plot      Yoko_kvo_georef.tif                                      1.000

Units with <95% geometric RGB/ROI coverage: 2


,unit_type,unit_id,site,fold,spatial_group,mosaic,roi_mosaic_coverage_fraction
14,plot,Mixt2,MOS,0,SG015,Mos_5_6_transparent_mosaic_group1_georef.tif,0.0
38,plot,Mixt1,MOS,1,SG031,Mos_5_6_transparent_mosaic_group1_georef.tif,0.0


Full-chip exclusion geometry prepared; original NB01 spatial groups untouched.


## 03 — Candidate training-chip manifest (geometry only; no image read)

Generate deterministic non-overlapping training-chip candidates from outer-training circles and outer-training plots, plus partial crown centres. The manifest records chip width, source unit/group, site and whether the candidate is **fully clear** of every held-out reference plus 50 m. Candidate source attribution remains separate from pixel-level label reconciliation. Sampling caps prevent excessive Drive I/O.

For Model A, plots are never training sources. For Model B, plot-labelled candidates are added. **Both use the same baseline circle and partial-crown candidates**, making training differences interpretable.


In [4]:
CHIP=256
MAX_CANDIDATES_PER_UNIT=40
MAX_PARTIAL_PER_SITE=240
SEED=20261118
from rasterio.windows import Window
def candidate_chip_centres(ds,geom,max_chips):
    # Build a bounded grid of chip centers inside source geometry. The chip itself
    # may protrude; full-window holdout exclusion is checked separately.
    in_ds=gpd.GeoSeries([geom],crs=REF_CRS).to_crs(ds.crs).iloc[0]
    raw=rasterio.windows.from_bounds(*in_ds.bounds,transform=ds.transform)
    step=CHIP
    rmin=max(CHIP//2,int(np.floor(raw.row_off)))
    cmin=max(CHIP//2,int(np.floor(raw.col_off)))
    rmax=min(ds.height-CHIP//2,int(np.ceil(raw.row_off+raw.height)))
    cmax=min(ds.width-CHIP//2,int(np.ceil(raw.col_off+raw.width)))
    if rmax<rmin or cmax<cmin:return []
    rrange=list(range(rmin,rmax+1,step)) or [rmin]
    crange=list(range(cmin,cmax+1,step)) or [cmin]
    pts=[]
    for rr in rrange:
        for cc in crange:
            xy=ds.xy(rr,cc)
            point=gpd.GeoSeries(gpd.points_from_xy([xy[0]],[xy[1]]),crs=ds.crs).to_crs(REF_CRS).iloc[0]
            if geom.intersects(point):pts.append((rr,cc))
    # Always offer centroid so narrow or sub-chip-sized units yield a candidate.
    xy=in_ds.representative_point()
    row,col=ds.index(xy.x,xy.y)
    if CHIP//2<=row<ds.height-CHIP//2 and CHIP//2<=col<ds.width-CHIP//2:
        pts.append((row,col))
    pts=sorted(set(pts))
    if len(pts)>max_chips:
        take=np.linspace(0,len(pts)-1,max_chips).astype(int)
        pts=[pts[i] for i in take]
    return pts
records=[]
for fold in range(3):
    fold_units=units[units.fold.ne(fold)].copy()
    for mosaic,d in mosaics.items():
        site=d['site']
        heldout=HELDOUT[fold][site]
        assigned=OWNERS[(OWNERS.mosaic==mosaic)&OWNERS.fold.ne(fold)]
        with rasterio.open(d['path']) as ds:
            for unit in assigned.itertuples(index=False):
                geom=fold_units[(fold_units.unit_type==unit.unit_type)&
                    (fold_units.unit_id==unit.unit_id)].geometry.iloc[0]
                for rr,cc in candidate_chip_centres(ds,geom,MAX_CANDIDATES_PER_UNIT):
                    win=Window(int(cc-CHIP//2),int(rr-CHIP//2),CHIP,CHIP)
                    bbox=box(*rasterio.windows.bounds(win,ds.transform))
                    bb=gpd.GeoSeries([bbox],crs=ds.crs).to_crs(REF_CRS).iloc[0]
                    records.append(dict(fold=fold,site=site,mosaic=mosaic,
                       row=int(win.row_off),col=int(win.col_off),
                       source='circle' if unit.unit_type=='circle' else 'plot',
                       source_id=str(unit.unit_id),spatial_group=str(unit.spatial_group),
                       blocked_by_outer_holdout=bool(bb.intersects(heldout)),
                       full_chip_area_m2=float(bb.area)))
            # Sample explicit partial crowns only outside holdout buffers.
            crown=partial[site]
            if len(crown):
                z=crown[crown.geometry.intersects(d['bounds'])].copy()
                # Stable spatial candidate selection, no held-out labels consulted.
                if len(z)>MAX_PARTIAL_PER_SITE:
                    z=z.sample(n=MAX_PARTIAL_PER_SITE,random_state=SEED+fold)
                for ix,part in z.iterrows():
                    xy=part.geometry.representative_point()
                    r,c=ds.index(xy.x,xy.y)
                    win=Window(int(c-CHIP//2),int(r-CHIP//2),CHIP,CHIP)
                    if (win.row_off<0 or win.col_off<0 or
                        win.col_off+CHIP>ds.width or win.row_off+CHIP>ds.height):continue
                    bb=gpd.GeoSeries([box(*rasterio.windows.bounds(win,ds.transform))],
                                     crs=ds.crs).to_crs(REF_CRS).iloc[0]
                    records.append(dict(fold=fold,site=site,mosaic=mosaic,
                       row=int(win.row_off),col=int(win.col_off),source='partial',
                       source_id=str(ix),spatial_group='partial',
                       blocked_by_outer_holdout=bool(bb.intersects(heldout)),
                       full_chip_area_m2=float(bb.area)))
CANDIDATES=pd.DataFrame(records)
assert not CANDIDATES.empty
# A mosaic/window may overlap several original annotations: keep the first in
# each source class for source balancing; labels will be constructed from geometry.
CANDIDATES=CANDIDATES.drop_duplicates(['fold','site','mosaic','row','col','source']).copy()
CANDIDATES['eligible']=~CANDIDATES.blocked_by_outer_holdout
print('Candidate windows, by fold/source and leakage status:')
display(CANDIDATES.groupby(['fold','source','eligible']).size().unstack(fill_value=0))
print('Candidate mosaic attribution:')
display(CANDIDATES.loc[CANDIDATES.eligible].groupby(['fold','site','source']).size().unstack(fill_value=0))
if CANDIDATES.loc[CANDIDATES.eligible].empty:
    raise RuntimeError('No training candidates after holdout exclusion')
assert not CANDIDATES.loc[CANDIDATES.eligible].blocked_by_outer_holdout.any()
CANDIDATES.to_csv(OUT/'02_full_chip_heldout_exclusion_manifest.csv',index=False)
print('Manifest:',OUT/'02_full_chip_heldout_exclusion_manifest.csv')
print('Important: this is a candidate manifest, NOT a guarantee of supervised pixel counts.')


Candidate windows, by fold/source and leakage status:


eligible      False  True 
fold source               
0    circle       0    538
     partial    200   1239
     plot         3    495
1    circle       0    537
     partial    146   1291
     plot         0    462
2    circle       0    523
     partial    238   1198
     plot         4    438

Candidate mosaic attribution:


source         circle  partial  plot
fold site                           
0    Baego        143      171    16
     MOS           75      650    40
     Yangambi     200      218   379
     Yoko         120      200    60
1    Baego        144      220    18
     MOS           73      657    38
     Yangambi     160      210   358
     Yoko         160      204    48
2    Baego        117      162    12
     MOS           86      649    32
     Yangambi     200      190   348
     Yoko         120      197    46

Manifest: /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/v4/02_rgb_circles_plus_plots_v1/02_full_chip_heldout_exclusion_manifest.csv
Important: this is a candidate manifest, NOT a guarantee of supervised pixel counts.


## 04 — Pre-training acceptance gate

The two models must share every eligible circle/partial training candidate, while Model B may *add* plot candidates. Geometry overlap and class contradictions are reported. **This is not yet a final pixel-label training cache**: next NB02 revision must perform limited RGB-window reads, rasterize source-specific positive/negative supervision, count valid positive and negative pixels by site/fold/source, reconcile conflicting annotations, and prepare a frozen train/inner-validation split.

Do **not** start GPU training just because this vector-level candidate audit passes. First inspect the circle/plot assignment, overlap disagreement, and candidate survival across all folds; then we can build and check real masked pixels once, using these manifests.


In [5]:
# Freeze a CPU-only gate report; explicit 'NOT TRAIN-READY' status prevents accidental use.
assert len(GEOMS)==108 and GEOMS.spatial_group.nunique()>0
assert len(OWNERS)==108
assert CANDIDATES.eligible.any()
assert set(CANDIDATES.source).issubset({'circle','plot','partial'})
assert CANDIDATES.groupby('fold').eligible.sum().gt(0).all()
train_counts=CANDIDATES.loc[CANDIDATES.eligible].groupby(['fold','source']).size().unstack(fill_value=0)
assert all(k in train_counts.columns for k in ['circle','plot','partial']),train_counts.columns
assert train_counts[['circle','plot','partial']].min().min()>0,'Insufficient sources in at least one fold'
manifest=dict(schema='musanga_v4_nb02_vector_preflight_v1',
  status='CPU_GEOMETRY_PREFLIGHT_COMPLETE_PIXEL_LABEL_AUDIT_PENDING',
  nb01_csv=str(FOLD_CSV),nb01_csv_sha256=SPLIT_CONTRACT['csv_sha256'],
  original_circle_folds_preserved=True,
  n_reference_units=108,n_spatial_groups=int(GEOMS.spatial_group.nunique()),
  n_folds=3,full_chip_px=CHIP,holdout_buffer_m=50,
  model_A='KVO_complete_circles_plus_partial_tree_crowns',
  model_B='Model_A_plus_Nadine_exhaustive_field_plots',
  sampling='same circle/partial candidates in both arms; extra plot candidates only in B',
  explicit_pending_checks=[
    'Inspect low RGB/ROI reference-unit coverage',
    'Approve plot unannotated background as negative or restrict to crown objects',
    'Resolve circle-vs-plot exhaustive annotation conflicts before pixel supervision',
    'Rasterize masks and count positive/negative and source support',
    'Enforce full-chip holdout filtering on final train AND inner selection',
    'Restrict plot and circle annotations to outer training groups',
    'Fix 256px model context and harmonized augmentation/optimizer',
    'Choose inner spatial selection without outer-fold outcome tuning'],
  candidates_csv=str(OUT/'02_full_chip_heldout_exclusion_manifest.csv'),
  candidates_sha256=hashlib.sha256((OUT/'02_full_chip_heldout_exclusion_manifest.csv').read_bytes()).hexdigest(),
  reference_overlap_audit=str(OUT/'02_reference_annotation_overlap_audit.json'))
(OUT/'02_cpu_gate_contract.json').write_text(json.dumps(manifest,indent=2))
display(train_counts)
print('NB02 STAGE 1 COMPLETE: GPU TRAINING INTENTIONALLY NOT ENABLED')
print('Saved:',OUT/'02_cpu_gate_contract.json')
print('Review outputs before generating supervised-pixel caches and training checkpoints.')


source,circle,partial,plot
fold,,,
0,538,1239,495
1,537,1291,462
2,523,1198,438


NB02 STAGE 1 COMPLETE: GPU TRAINING INTENTIONALLY NOT ENABLED
Saved: /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/v4/02_rgb_circles_plus_plots_v1/02_cpu_gate_contract.json
Review outputs before generating supervised-pixel caches and training checkpoints.
